**Before Using this Notebook, delete all Files which are in thesis/data!**

In [20]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [21]:
import sys, pathlib

# adjust this to your repo root if different
repo_root = pathlib.Path.cwd()

repo_root = repo_root.parent.parent

sys.path.insert(0, str(repo_root))
print('Inserted repo root into sys.path:', repo_root)

Inserted repo root into sys.path: /Users/fabian/qecsim


In [22]:
import sinter
from matplotlib import pyplot as plt
from matplotlib.ticker import LogFormatterMathtext, FuncFormatter
import os

In [23]:
# Modify Plots to be IN latex Sytle
plt.rcParams.update({
    "text.usetex": True,
    "font.family": "serif",
    "font.serif": ["Computer Modern"],
    "font.size": 14,
    "axes.labelsize": 16,
    "xtick.labelsize": 13,
    "ytick.labelsize": 13,
    "legend.fontsize": 12,
    "legend.title_fontsize": 13,
})

# Define Sci plotter for later use in all plots
def sci_format(x, pos):
    # This takes the number, divides by 1e-3, and formats it nicely
    # -> Need to be done since the formatting is shit otherwise
    return rf"${x / 1e-3:g} \times 10^{{-3}}$"

In [27]:
cases = [
    ("X+", "X"), 
    ("Y+", "Y"), 
    ("Z0", "Z")
]

THRESHOLD_P = 4.06e-3

for curr_config in cases:
    config_suffix = '_'.join(curr_config)
    csv_filename = f"../data/stats_memory_{config_suffix}.csv"
    plot_filename = f"../plots/plot_memory_{config_suffix}.pdf"

    # Check if the data file exists before trying to plot
    if not os.path.exists(csv_filename):
        print(f"Skipping {csv_filename}: File not found.")
        continue

    # 1. Load the stats from the CSV
    stats = sinter.read_stats_from_csv_files(csv_filename)

    # 2. Create the figure
    fig, ax = plt.subplots(1, 1, figsize=(7, 5))

    # 3. Use sinter's plotting utility
    sinter.plot_error_rate(
        ax=ax,
        stats=stats,
        x_func=lambda s: s.json_metadata['p'],
        group_func=lambda s: s.json_metadata['d'],
    )

    # 4. Setup Plot
    ax.loglog()
    ax.xaxis.set_major_formatter(LogFormatterMathtext())
    ax.yaxis.set_major_formatter(LogFormatterMathtext())
    ax.set_xlim(1e-3, 0.7e-2)
    ax.set_ylim(1e-6, 1.0)
    ax.set_title(rf"Surface Memory Circuit-Noise: {curr_config[0]} $\rightarrow$ {curr_config[1]}", fontsize=20)
    ax.set_xlabel("Physical Error Rate $p$")
    ax.set_ylabel("Logical Error Rate $p_L$")
    ax.grid(which='major', linestyle='-', alpha=0.6)
    ax.grid(which='minor', linestyle=':', alpha=0.3)

    # Threshold Line
    ax.axvline(x=THRESHOLD_P, color='dimgray', linestyle='--', alpha=0.8)
    
    # Adding Legend
    ax.legend(title="Distance $d$", loc='upper left')

    ###################### INSERT THRESHOLD ZOOM ########################
    # Position: [x0, y0, width, height] in normalized coordinates (0 to 1)
    # Placing it in the upper left [0.05, 0.5] to avoid overlapping the curve and legend
    axins = ax.inset_axes([0.58, 0.18, 0.35, 0.4])

    # Plot the exact same data onto the inset axes
    sinter.plot_error_rate(
        ax=axins,
        stats=stats,
        x_func=lambda s: s.json_metadata['p'],
        group_func=lambda s: s.json_metadata['d'],
    )

    # Styling and zooming the inset
    axins.loglog()
    axins.set_xticks([3e-3, 4e-3, 5e-3])
    axins.xaxis.set_major_formatter(FuncFormatter(sci_format))
    axins.xaxis.set_minor_formatter(plt.NullFormatter())
    axins.tick_params(axis='x', rotation=45, labelsize=10)
    
    # Zoomed limits for x (around p = 4e-3)
    axins.set_xlim(3e-3, 5e-3)
    
    # You might need to tweak the y-limit depending on where your actual threshold crossing occurs
    axins.set_ylim(1e-2, 1e-1) 
    
    axins.grid(which='major', linestyle='-', alpha=0.6)
    axins.grid(which='minor', linestyle=':', alpha=0.3)

    #Adding Threshold line to sub-plot
    axins.axvline(x=THRESHOLD_P, color='dimgray', linestyle='--', alpha=0.8)
    
    # Remove titles/labels/legends from the inset so it stays clean
    axins.set_xlabel('')
    axins.set_ylabel('')
    if axins.get_legend():
        axins.get_legend().remove()

    # Automatically draw a box and connecting lines from the zoom area to the inset
    ax.indicate_inset_zoom(axins, edgecolor="black")
    ########################################################################

    # 5. Save the plot
    fig.set_dpi(300)
    plt.tight_layout()
    plt.savefig(plot_filename, bbox_inches='tight')
    plt.close(fig)  # Close to free up memory during the loop

    print(f"Successfully saved: {plot_filename}")

Successfully saved: ../plots/plot_memory_X+_X.pdf
Successfully saved: ../plots/plot_memory_Y+_Y.pdf
Successfully saved: ../plots/plot_memory_Z0_Z.pdf


In [28]:
cases = [
    ("I0", "Z0", "Z", "Z"), 
    ("Z0", "I0", "Z", "I"), 
    ("X+", "I0", "X", "X"), 
    ("I0", "X+", "I", "X")
]

THRESHOLD_P = 2.8e-3

for curr_config in cases:
    config_suffix = '_'.join(curr_config)
    csv_filename = f"../data/stats_surgery_{config_suffix}.csv"
    plot_filename = f"../plots/plot_surgery_{config_suffix}.pdf"

    # Check if the data file exists before trying to plot
    if not os.path.exists(csv_filename):
        print(f"Skipping {csv_filename}: File not found.")
        continue

    # 1. Load the stats from the CSV
    stats = sinter.read_stats_from_csv_files(csv_filename)

    # 2. Create the figure
    fig, ax = plt.subplots(1, 1, figsize=(7, 5))

    # 3. Use sinter's plotting utility
    sinter.plot_error_rate(
        ax=ax,
        stats=stats,
        x_func=lambda s: s.json_metadata['p'],
        group_func=lambda s: s.json_metadata['d'],
    )

    # 4. Setup Plot
    ax.loglog()
    ax.xaxis.set_major_formatter(LogFormatterMathtext())
    ax.yaxis.set_major_formatter(LogFormatterMathtext())
    ax.set_ylim(1e-3, 1e-0)
    ax.set_xlim(1e-3,0.7e-2)
    ax.set_title(rf"Lattice Surgery Circuit-Noise: {curr_config[0]} {curr_config[1]} $\rightarrow$ {curr_config[2]} {curr_config[3]}", fontsize=20)
    ax.set_xlabel("Physical Error Rate $p$")
    ax.set_ylabel("Logical Error Rate $p_L$")
    ax.grid(which='major', linestyle='-', alpha=0.6)
    ax.grid(which='minor', linestyle=':', alpha=0.3)

    # Threshold Line
    ax.axvline(x=THRESHOLD_P, color='dimgray', linestyle='--', alpha=0.8)

    ax.legend(title="Distance $d$", loc='upper left')


    ###################### INSERT THRESHOLD ZOOM ########################
    # Position: [x0, y0, width, height] in normalized coordinates (0 to 1)
    # Placing it in the upper left [0.05, 0.5] to avoid overlapping the curve and legend
    axins = ax.inset_axes([0.57, 0.18, 0.35, 0.4])

    # Plot the exact same data onto the inset axes
    sinter.plot_error_rate(
        ax=axins,
        stats=stats,
        x_func=lambda s: s.json_metadata['p'],
        group_func=lambda s: s.json_metadata['d'],
    )

    # Styling and zooming the inset
    axins.loglog()
    axins.set_xticks([2e-3, 3e-3, 4e-3])
    axins.xaxis.set_major_formatter(FuncFormatter(sci_format))
    axins.xaxis.set_minor_formatter(plt.NullFormatter())

    # 3. Rotate the labels 45 degrees and slightly reduce font size to ensure they fit
    axins.tick_params(axis='x', rotation=45, labelsize=10)
    
    # Zoomed limits for x (around p = 4e-3)
    axins.set_xlim(2e-3, 4e-3)
    
    # You might need to tweak the y-limit depending on where your actual threshold crossing occurs
    axins.set_ylim(1e-1, 6e-1) 
    
    axins.grid(which='major', linestyle='-', alpha=0.6)
    axins.grid(which='minor', linestyle=':', alpha=0.3)

    #Adding Threshold line to sub-plot
    axins.axvline(x=THRESHOLD_P, color='dimgray', linestyle='--', alpha=0.8)
    
    # Remove titles/labels/legends from the inset so it stays clean
    axins.set_xlabel('')
    axins.set_ylabel('')
    if axins.get_legend():
        axins.get_legend().remove()

    # Automatically draw a box and connecting lines from the zoom area to the inset
    ax.indicate_inset_zoom(axins, edgecolor="black")
    ########################################################################

    # 5. Save the plot
    fig.set_dpi(300)
    plt.tight_layout()
    plt.savefig(plot_filename, bbox_inches='tight')
    plt.close(fig)  # Close to free up memory during the loop

    print(f"Successfully saved: {plot_filename}")

Successfully saved: ../plots/plot_surgery_I0_Z0_Z_Z.pdf
Successfully saved: ../plots/plot_surgery_Z0_I0_Z_I.pdf
Successfully saved: ../plots/plot_surgery_X+_I0_X_X.pdf
Successfully saved: ../plots/plot_surgery_I0_X+_I_X.pdf
